# ⭐ Day 80: Advanced Feature Engineering for Telecom Customer Churn Prediction

### *Day 80 of the 369-Day Python & AI Learning Path* 🚀

---

**Creating High-Impact Features to Supercharge Churn Model Performance**

## 🎯 Introduction

Welcome to **Day 80** of your transformative 369-day Python & AI Learning Path! Today, we dive deep into one of the most critical skills in real-world Machine Learning: **Advanced Feature Engineering**.

In the competitive telecom industry, customer churn prediction is a multi-billion-dollar challenge. While algorithm selection matters, **feature engineering often delivers the biggest performance gains**. Raw data is rarely predictive in its original form — the magic happens when we transform it into meaningful signals that capture customer behavior, financial stress, and loyalty patterns.

### What You Will Learn Today:

✅ **Temporal & Tenure Features** — Extract time-based insights from signup dates  
✅ **Usage Behavior & Monetary Features** — Build ratios, efficiency metrics, and risk flags  
✅ **Customer Profile & Interaction Features** — Segment customers and create powerful interactions  
✅ **Contract, Support & Loyalty Features** — Quantify commitment and dissatisfaction signals  
✅ **Text Features from Feedback** — Mine unstructured customer feedback for predictive keywords  
✅ **Feature Correlation Analysis** — Visualize which engineered features drive churn most strongly  

By the end of this notebook, you will have transformed a basic dataset into a **rich, feature-engineered goldmine** ready for advanced modeling. Let's engineer some winning features! 💪

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set professional styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
%matplotlib inline

print("✅ All libraries imported successfully!")

## 📊 Dataset Overview

In [ ]:
# Load the telecom customer churn dataset
df = pd.read_csv('/home/workdir/attachments/telecom_customer_churn_feature_engineering.csv')

print(f"📊 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n🔍 Column Names:\n{list(df.columns)}")
print(f"\n💾 Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
df.head()

In [ ]:
# Quick profile of the dataset
print("=" * 60)
print("📈 CHURN DISTRIBUTION")
print("=" * 60)
churn_dist = df['churn'].value_counts(normalize=True)
print(churn_dist)
print(f"\n🎯 Churn Rate: {churn_dist.get(1, churn_dist.get('Yes', 0)):.2%}")

print("\n" + "=" * 60)
print("📋 DATA TYPES & MISSING VALUES")
print("=" * 60)
dtype_summary = pd.DataFrame({
    'Data Type': df.dtypes,
    'Non-Null Count': df.count(),
    'Missing Values': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
print(dtype_summary)

---

## 1️⃣ Temporal & Tenure Features 📅

**Why tenure matters:** Customer tenure is one of the strongest predictors of churn. New customers are significantly more likely to churn than long-tenured ones. By engineering multiple tenure-related features, we capture different dimensions of customer longevity and seasonal signup patterns.

In [ ]:
# Convert signup_date to datetime for temporal feature extraction
df['signup_date'] = pd.to_datetime(df['signup_date'])

# Define reference date (day after the most recent signup) for tenure calculation
current_date = df['signup_date'].max() + pd.Timedelta(days=1)
print(f"📅 Reference Date for Tenure Calculation: {current_date.date()}")

# Core tenure features
df['tenure_days'] = (current_date - df['signup_date']).dt.days
df['tenure_months'] = df['tenure_days'] // 30
df['tenure_years'] = df['tenure_days'] / 365.25

# Seasonal and cyclical temporal features
df['signup_month'] = df['signup_date'].dt.month
df['signup_year'] = df['signup_date'].dt.year
df['signup_dayofweek'] = df['signup_date'].dt.dayofweek  # 0=Monday, 6=Sunday
df['signup_quarter'] = df['signup_date'].dt.quarter
df['signup_is_weekend'] = (df['signup_dayofweek'] >= 5).astype(int)

# Tenure segmentation (business-critical bins)
df['tenure_segment'] = pd.cut(
    df['tenure_months'],
    bins=[-1, 6, 12, 24, 36, 999],
    labels=['0-6 months', '6-12 months', '1-2 years', '2-3 years', '3+ years']
)

print("✅ Temporal and tenure features created successfully!")
print(f"\n📊 Tenure Summary (Months):")
print(df['tenure_months'].describe())

df[['signup_date', 'tenure_days', 'tenure_months', 'tenure_years', 'tenure_segment']].head(10)

In [ ]:
# Visualize tenure distribution and its relationship with churn
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Tenure distribution
sns.histplot(data=df, x='tenure_months', hue='churn', bins=30, kde=True, ax=axes[0], alpha=0.7)
axes[0].set_title('📊 Tenure Distribution by Churn Status', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tenure (Months)')
axes[0].set_ylabel('Count')
axes[0].legend(['Retained', 'Churned'])

# Churn rate by tenure segment
tenure_churn = df.groupby('tenure_segment')['churn'].agg(['mean', 'count']).reset_index()
tenure_churn['churn_rate'] = tenure_churn['mean']
sns.barplot(data=tenure_churn, x='tenure_segment', y='churn_rate', palette='coolwarm', ax=axes[1])
axes[1].set_title('📉 Churn Rate by Tenure Segment', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Tenure Segment')
axes[1].set_ylabel('Churn Rate')
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(tenure_churn['churn_rate']):
    axes[1].text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🔍 Insight: New customers (0-6 months) show the highest churn risk!")

---

## 2️⃣ Usage Behavior & Monetary Features 💰

**Why behavior ratios matter:** Raw usage numbers tell only part of the story. Ratios like *bill-to-income* reveal financial stress, while *usage efficiency* shows whether customers feel they're getting value for money. These relative features often outperform raw counts in predictive models.

In [ ]:
# Financial stress and behavioral ratios
df['bill_to_income_ratio'] = df['monthly_bill'] / (df['monthly_income'] + 1)  # +1 to avoid div by zero
df['gb_per_call_minute'] = df['internet_usage_gb'] / (df['call_minutes'] + 1)
df['usage_efficiency'] = (df['internet_usage_gb'] + df['call_minutes'] / 60) / (df['monthly_bill'] + 1)
df['income_per_usage'] = df['monthly_income'] / (df['internet_usage_gb'] + df['call_minutes'] / 60 + 1)

# High-usage flags (top quartile indicators)
df['high_internet_user'] = (df['internet_usage_gb'] > df['internet_usage_gb'].quantile(0.75)).astype(int)
df['high_call_user'] = (df['call_minutes'] > df['call_minutes'].quantile(0.75)).astype(int)
df['high_bill_user'] = (df['monthly_bill'] > df['monthly_bill'].quantile(0.75)).astype(int)
df['low_income_flag'] = (df['monthly_income'] < df['monthly_income'].quantile(0.25)).astype(int)

# Critical risk flag: High bill but low usage (value perception issue)
df['high_bill_low_usage'] = (
    (df['high_bill_user'] == 1) & 
    (df['high_internet_user'] == 0) & 
    (df['high_call_user'] == 0)
.astype(int)

# Value perception score (higher = better value for money)
df['value_perception_score'] = (
    (df['internet_usage_gb'] / (df['monthly_bill'] + 1)) * 10 +
    (df['call_minutes'] / (df['monthly_bill'] + 1))
)

print("✅ Usage behavior and monetary features created successfully!")
print(f"\n💡 High Bill + Low Usage Risk Flag: {df['high_bill_low_usage'].sum()} customers flagged")
print(f"📊 Average Bill-to-Income Ratio: {df['bill_to_income_ratio'].mean():.3f}")

# Display key monetary features
monetary_cols = ['monthly_bill', 'monthly_income', 'bill_to_income_ratio', 
                 'usage_efficiency', 'high_bill_low_usage']
df[monetary_cols].head(10)

In [ ]:
# Visualize monetary features and churn relationship
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Bill-to-income ratio by churn
sns.boxplot(data=df, x='churn', y='bill_to_income_ratio', ax=axes[0,0], palette='coolwarm')
axes[0,0].set_title('💰 Bill-to-Income Ratio by Churn Status', fontweight='bold')
axes[0,0].set_xticklabels(['Retained', 'Churned'])

# Usage efficiency by churn
sns.boxplot(data=df, x='churn', y='usage_efficiency', ax=axes[0,1], palette='coolwarm')
axes[0,1].set_title('⚡ Usage Efficiency by Churn Status', fontweight='bold')
axes[0,1].set_xticklabels(['Retained', 'Churned'])

# High bill low usage flag
risk_crosstab = pd.crosstab(df['high_bill_low_usage'], df['churn'], normalize='index')
risk_crosstab.plot(kind='bar', ax=axes[1,0], color=['#2ecc71', '#e74c3c'])
axes[1,0].set_title('🚨 Churn Rate: High Bill + Low Usage Flag', fontweight='bold')
axes[1,0].set_xlabel('High Bill + Low Usage (0=No, 1=Yes)')
axes[1,0].set_ylabel('Proportion')
axes[1,0].legend(['Retained', 'Churned'])
axes[1,0].tick_params(axis='x', rotation=0)

# Value perception score distribution
sns.histplot(data=df, x='value_perception_score', hue='churn', bins=30, kde=True, ax=axes[1,1], alpha=0.6)
axes[1,1].set_title('📈 Value Perception Score Distribution', fontweight='bold')
axes[1,1].legend(['Retained', 'Churned'])

plt.tight_layout()
plt.show()

---

## 3️⃣ Customer Profile & Interaction Features 👥

**Why interactions matter:** Combining demographic variables creates powerful interaction features. For example, a young customer with a high income behaves very differently from a senior on a fixed income. City-level churn rates also capture geographic market dynamics.

In [ ]:
# Age segmentation with business-relevant brackets
df['age_group'] = pd.cut(
    df['age'], 
    bins=[0, 25, 35, 45, 55, 100], 
    labels=['18-25', '26-35', '36-45', '46-55', '56+']
)

# Income quintile segmentation
df['income_segment'] = pd.qcut(
    df['monthly_income'], 
    q=5, 
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
)

# Education + Employment interaction (socioeconomic profile)
df['edu_employment'] = df['education_level'].astype(str) + '_' + df['employment_status'].astype(str)

# City-level churn rate (geographic risk indicator)
city_churn = df.groupby('city')['churn'].mean()
df['city_churn_rate'] = df['city'].map(city_churn)

# Age-income interaction (life stage indicator)
df['age_income_profile'] = df['age_group'].astype(str) + '_' + df['income_segment'].astype(str)

# Customer lifetime value proxy
df['clv_proxy'] = df['tenure_months'] * df['monthly_bill']

print("✅ Customer profile and interaction features created successfully!")
print(f"\n🌍 Number of unique cities: {df['city'].nunique()}")
print(f"👔 Unique Education-Employment combinations: {df['edu_employment'].nunique()}")
print(f"\n📊 City Churn Rate Statistics:")
print(df['city_churn_rate'].describe())

df[['age', 'age_group', 'monthly_income', 'income_segment', 'edu_employment', 'city_churn_rate']].head(10)

In [ ]:
# Visualize customer profile features
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Churn rate by age group
age_churn = df.groupby('age_group')['churn'].mean().reset_index()
sns.barplot(data=age_churn, x='age_group', y='churn', palette='viridis', ax=axes[0,0])
axes[0,0].set_title('👥 Churn Rate by Age Group', fontweight='bold')
axes[0,0].set_ylabel('Churn Rate')
for i, v in enumerate(age_churn['churn']):
    axes[0,0].text(i, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')

# Churn rate by income segment
income_churn = df.groupby('income_segment')['churn'].mean().reset_index()
sns.barplot(data=income_churn, x='income_segment', y='churn', palette='plasma', ax=axes[0,1])
axes[0,1].set_title('💵 Churn Rate by Income Segment', fontweight='bold')
axes[0,1].set_ylabel('Churn Rate')
axes[0,1].tick_params(axis='x', rotation=15)
for i, v in enumerate(income_churn['churn']):
    axes[0,1].text(i, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')

# City churn rate distribution
sns.histplot(df['city_churn_rate'], bins=20, kde=True, ax=axes[1,0], color='coral')
axes[1,0].set_title('🌍 Distribution of City Churn Rates', fontweight='bold')
axes[1,0].set_xlabel('City Churn Rate')
axes[1,0].axvline(df['city_churn_rate'].mean(), color='red', linestyle='--', 
                  label=f'Mean: {df["city_churn_rate"].mean():.2%}')
axes[1,0].legend()

# CLV Proxy by churn
sns.boxplot(data=df, x='churn', y='clv_proxy', ax=axes[1,1], palette='coolwarm')
axes[1,1].set_title('💎 Customer Lifetime Value Proxy by Churn', fontweight='bold')
axes[1,1].set_xticklabels(['Retained', 'Churned'])
axes[1,1].set_ylabel('CLV Proxy ($)')

plt.tight_layout()
plt.show()

---

## 4️⃣ Contract, Support & Loyalty Features 📜

**Why contract features matter:** Contract type is a direct proxy for customer commitment. Monthly customers can leave anytime, while annual contract customers face switching costs. Support ticket frequency indicates dissatisfaction — a leading churn indicator.

In [ ]:
# Contract duration mapping
contract_map = {'Monthly': 1, '6-Month': 6, '12-Month': 12, '24-Month': 24}
df['contract_months'] = df['contract_type'].map(contract_map)

# Support ticket metrics (normalized by tenure)
df['support_ticket_rate'] = df['support_tickets'] / (df['tenure_months'] + 1)
df['high_support_tickets'] = (df['support_tickets'] >= 3).astype(int)
df['support_ticket_intensity'] = pd.cut(
    df['support_tickets'],
    bins=[-1, 0, 1, 3, 999],
    labels=['None', 'Low', 'Medium', 'High']
)

# Loyalty interaction features
df['tenure_contract_interaction'] = df['tenure_months'] * df['contract_months']
df['contract_value'] = df['contract_months'] * df['monthly_bill']
df['is_long_term_contract'] = (df['contract_months'] >= 12).astype(int)

# Loyalty score composite
df['loyalty_score'] = (
    df['tenure_months'] * 0.3 +
    df['contract_months'] * 0.4 +
    (1 - df['support_ticket_rate']) * 0.3
)

print("✅ Contract, support, and loyalty features created successfully!")
print(f"\n📜 Contract Type Distribution:")
print(df['contract_type'].value_counts())
print(f"\n🎫 Support Ticket Statistics:")
print(df['support_tickets'].describe())
print(f"\n⭐ Average Loyalty Score: {df['loyalty_score'].mean():.2f}")

df[['contract_type', 'contract_months', 'support_tickets', 'support_ticket_rate', 
    'tenure_contract_interaction', 'loyalty_score']].head(10)

In [ ]:
# Visualize contract and support features
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Churn rate by contract type
contract_churn = df.groupby('contract_type')['churn'].mean().reset_index()
sns.barplot(data=contract_churn, x='contract_type', y='churn', palette='RdYlGn_r', ax=axes[0,0])
axes[0,0].set_title('📜 Churn Rate by Contract Type', fontweight='bold')
axes[0,0].set_ylabel('Churn Rate')
for i, v in enumerate(contract_churn['churn']):
    axes[0,0].text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

# Support tickets vs churn
support_churn = df.groupby('support_ticket_intensity')['churn'].mean().reset_index()
sns.barplot(data=support_churn, x='support_ticket_intensity', y='churn', 
            palette='Reds', ax=axes[0,1])
axes[0,1].set_title('🎫 Churn Rate by Support Ticket Intensity', fontweight='bold')
axes[0,1].set_ylabel('Churn Rate')
for i, v in enumerate(support_churn['churn']):
    axes[0,1].text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

# Loyalty score distribution
sns.histplot(data=df, x='loyalty_score', hue='churn', bins=30, kde=True, ax=axes[1,0], alpha=0.6)
axes[1,0].set_title('⭐ Loyalty Score Distribution by Churn', fontweight='bold')
axes[1,0].legend(['Retained', 'Churned'])

# Tenure-Contract interaction
sns.scatterplot(data=df.sample(min(1000, len(df))), x='tenure_months', y='contract_months', 
                hue='churn', alpha=0.6, ax=axes[1,1], palette='coolwarm')
axes[1,1].set_title('🔗 Tenure vs Contract Months (Colored by Churn)', fontweight='bold')
axes[1,1].legend(['Retained', 'Churned'])

plt.tight_layout()
plt.show()

---

## 5️⃣ Text Features from Customer Feedback 💬

**Why text features matter:** Customer feedback contains unstructured but highly predictive signals. Keyword-based features can capture billing complaints, service quality issues, and positive sentiment — all strong churn predictors when properly extracted.

In [ ]:
# Preprocess feedback text
feedback = df['customer_feedback'].astype(str).str.lower().fillna('')

# Keyword-based feature extraction
df['billing_issue'] = feedback.str.contains(r'bill|billing|expensive|overcharge|fee', regex=True).astype(int)
df['coverage_issue'] = feedback.str.contains(r'coverage|poor|drop|signal|dead zone', regex=True).astype(int)
df['speed_issue'] = feedback.str.contains(r'slow|speed|lag|buffer|loading', regex=True).astype(int)
df['service_issue'] = feedback.str.contains(r'service|support|rude|unhelpful|wait', regex=True).astype(int)
df['positive_feedback'] = feedback.str.contains(r'happy|satisfied|good|great|excellent|helpful|love', regex=True).astype(int)
df['complaint_count'] = df['billing_issue'] + df['coverage_issue'] + df['speed_issue'] + df['service_issue']
df['has_any_complaint'] = (df['complaint_count'] > 0).astype(int)
df['sentiment_ratio'] = df['positive_feedback'] - df['has_any_complaint']

print("✅ Text features extracted from customer feedback successfully!")
print(f"\n💬 Feedback Keyword Analysis:")
print(f"   Billing Issues: {df['billing_issue'].sum():,} customers ({df['billing_issue'].mean():.1%})")
print(f"   Coverage Issues: {df['coverage_issue'].sum():,} customers ({df['coverage_issue'].mean():.1%})")
print(f"   Speed Issues: {df['speed_issue'].sum():,} customers ({df['speed_issue'].mean():.1%})")
print(f"   Service Issues: {df['service_issue'].sum():,} customers ({df['service_issue'].mean():.1%})")
print(f"   Positive Feedback: {df['positive_feedback'].sum():,} customers ({df['positive_feedback'].mean():.1%})")
print(f"   Any Complaint: {df['has_any_complaint'].sum():,} customers ({df['has_any_complaint'].mean():.1%})")

# Display sample feedback with extracted features
text_cols = ['customer_feedback', 'billing_issue', 'coverage_issue', 
             'speed_issue', 'positive_feedback', 'complaint_count']
df[text_cols].head(10)

In [ ]:
# Visualize text feature impact on churn
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

text_features = ['billing_issue', 'coverage_issue', 'speed_issue', 
                 'service_issue', 'positive_feedback', 'has_any_complaint']
titles = ['💰 Billing Issue', '📡 Coverage Issue', '⚡ Speed Issue', 
          '🎧 Service Issue', '😊 Positive Feedback', '⚠️ Any Complaint']
colors = ['#e74c3c', '#e67e22', '#f39c12', '#9b59b6', '#2ecc71', '#e74c3c']

for idx, (feature, title, color) in enumerate(zip(text_features, titles, colors)):
    crosstab = pd.crosstab(df[feature], df['churn'], normalize='index')
    crosstab.plot(kind='bar', ax=axes[idx], color=['#2ecc71', '#e74c3c'], width=0.6)
    axes[idx].set_title(title, fontweight='bold', fontsize=12)
    axes[idx].set_xlabel(f'{feature.replace("_", " ").title()} (0=No, 1=Yes)')
    axes[idx].set_ylabel('Proportion')
    axes[idx].legend(['Retained', 'Churned'])
    axes[idx].tick_params(axis='x', rotation=0)
    axes[idx].set_ylim(0, 1)

plt.suptitle('📊 Churn Rate by Text-Derived Feedback Features', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 6️⃣ Feature Correlation Analysis 📊

**Why correlation analysis matters:** After engineering dozens of features, we need to understand which ones are most predictive of churn. This helps with feature selection, reduces multicollinearity, and guides model interpretation.

In [ ]:
# Identify all numeric columns for correlation analysis
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'churn' in numeric_cols:
    numeric_cols.remove('churn')

# Calculate correlation with churn
corr_with_churn = df[numeric_cols + ['churn']].corr()['churn'].drop('churn').sort_values(ascending=False)

print("=" * 70)
print("📈 TOP 15 FEATURES MOST CORRELATED WITH CHURN")
print("=" * 70)
top_positive = corr_with_churn.head(8)
top_negative = corr_with_churn.tail(7)

print("\n🚨 Highest Positive Correlation (Churn Drivers):")
for feat, corr in top_positive.items():
    print(f"   {feat:35s}: {corr:+.4f}")

print("\n✅ Highest Negative Correlation (Retention Drivers):")
for feat, corr in top_negative.items():
    print(f"   {feat:35s}: {corr:+.4f}")

print(f"\n📊 Total Engineered Numeric Features: {len(numeric_cols)}")

In [ ]:
# Correlation heatmap of top features
top_features = corr_with_churn.abs().sort_values(ascending=False).head(20).index.tolist() + ['churn']
corr_matrix = df[top_features].corr()

plt.figure(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    cmap='RdBu_r',
    center=0,
    fmt='.2f',
    linewidths=0.5,
    cbar_kws={'shrink': 0.8, 'label': 'Correlation Coefficient'},
    square=True
)
plt.title('🔥 Correlation Heatmap: Top 20 Engineered Features vs Churn', 
          fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance bar chart
plt.figure(figsize=(14, 10))
corr_sorted = corr_with_churn.reindex(corr_with_churn.abs().sort_values(ascending=True).index)

colors = ['#2ecc71' if x < 0 else '#e74c3c' for x in corr_sorted.values]
bars = plt.barh(range(len(corr_sorted)), corr_sorted.values, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)

plt.yticks(range(len(corr_sorted)), corr_sorted.index, fontsize=10)
plt.xlabel('Correlation with Churn', fontsize=12, fontweight='bold')
plt.title('📊 Feature Correlation with Churn: Complete Rankings', fontsize=16, fontweight='bold', pad=20)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, corr_sorted.values)):
    plt.text(val + (0.002 if val >= 0 else -0.002), i, f'{val:+.3f}', 
             va='center', ha='left' if val >= 0 else 'right', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🎯 Key Insight: Focus on the top positive and negative correlates for maximum model impact!")

---

## 🛠️ Hands-On Exercises

Now it's your turn to apply what you've learned! Complete the following exercises to deepen your feature engineering skills:

### Exercise 1: Create a Total Usage Score
Engineer a feature called `total_usage_score` that combines internet and call usage into a single metric. Use the formula:  
`total_usage_score = internet_usage_gb + (call_minutes / 10)`

### Exercise 2: Create an Age-Contract Interaction Feature
Build an interaction feature between `age_group` and `contract_type` to capture how different life stages interact with contract commitment levels.

### Exercise 3: Build a Composite Churn Risk Score
Create a `churn_risk_score` using at least 3 of the risk-related features engineered today. Assign weights to each component based on their business importance.

### Exercise 4: Create a Creative Custom Feature
Design one additional feature that you believe would be highly predictive for churn prediction. Explain your reasoning in a comment.

---

*Write your solutions in the code cells below!*

In [ ]:
# 📝 Exercise 1: Create total_usage_score
# Formula: internet_usage_gb + (call_minutes / 10)

# Your code here:




In [ ]:
# 📝 Exercise 2: Create age-contract interaction feature
# Combine age_group and contract_type into a single categorical feature

# Your code here:




In [ ]:
# 📝 Exercise 3: Build composite churn_risk_score
# Use at least 3 risk features with custom weights

# Your code here:




In [ ]:
# 📝 Exercise 4: Create your own creative feature
# Explain your reasoning in a comment

# Your code here:




---

## ✅ Solutions

Here are the complete solutions for the hands-on exercises. Compare your approach with these implementations!

In [ ]:
# ✅ Solution 1: Total Usage Score
# Combines internet and voice usage into a unified usage metric
df['total_usage_score'] = df['internet_usage_gb'] + (df['call_minutes'] / 10)

print("✅ Exercise 1 Complete: Total Usage Score created!")
print(f"📊 Statistics:")
print(df['total_usage_score'].describe())
print(f"\n🔍 Sample values:")
print(df[['internet_usage_gb', 'call_minutes', 'total_usage_score']].head())

In [ ]:
# ✅ Solution 2: Age-Contract Interaction Feature
# Captures how life stage interacts with contract commitment
df['age_contract_interaction'] = df['age_group'].astype(str) + "_" + df['contract_type'].astype(str)

print("✅ Exercise 2 Complete: Age-Contract Interaction created!")
print(f"\n📋 Unique combinations: {df['age_contract_interaction'].nunique()}")
print(f"\n📊 Top combinations by frequency:")
print(df['age_contract_interaction'].value_counts().head(10))

# Show churn rate by this interaction
interaction_churn = df.groupby('age_contract_interaction')['churn'].agg(['mean', 'count']).reset_index()
interaction_churn = interaction_churn[interaction_churn['count'] >= 10].sort_values('mean', ascending=False)
print(f"\n🚨 Highest churn rate combinations (min 10 customers):")
print(interaction_churn.head())

In [ ]:
# ✅ Solution 3: Composite Churn Risk Score
# Weighted composite using multiple risk signals
# Rationale: High bill + low usage is the strongest risk (weight 3)
# Support tickets indicate dissatisfaction (weight 2)
# Billing issues directly drive churn (weight 2)
# Lack of positive feedback is a warning sign (weight 1)

df['churn_risk_score'] = (
    df['high_bill_low_usage'] * 3 +
    df['high_support_tickets'] * 2 +
    df['billing_issue'] * 2 +
    (1 - df['positive_feedback']) * 1 +
    df['coverage_issue'] * 1 +
    df['speed_issue'] * 1
)

print("✅ Exercise 3 Complete: Composite Churn Risk Score created!")
print(f"\n📊 Risk Score Distribution:")
print(df['churn_risk_score'].value_counts().sort_index())

# Validate: show churn rate by risk score
risk_validation = df.groupby('churn_risk_score')['churn'].agg(['mean', 'count']).reset_index()
risk_validation.columns = ['churn_risk_score', 'churn_rate', 'customer_count']
print(f"\n🎯 Validation: Churn Rate by Risk Score:")
print(risk_validation)

# Visualize
plt.figure(figsize=(12, 6))
sns.barplot(data=risk_validation, x='churn_risk_score', y='churn_rate', palette='Reds')
plt.title('🎯 Churn Rate by Composite Risk Score', fontsize=14, fontweight='bold')
plt.xlabel('Churn Risk Score')
plt.ylabel('Churn Rate')
for i, row in risk_validation.iterrows():
    plt.text(i, row['churn_rate'] + 0.01, f"{row['churn_rate']:.1%}\n(n={row['customer_count']})", 
             ha='center', fontsize=9, fontweight='bold')
plt.ylim(0, 1)
plt.show()

In [ ]:
# ✅ Solution 4: Creative Custom Feature
# Feature: 'engagement_per_dollar' — measures how actively engaged a customer is relative to what they pay
# Rationale: Customers who pay a lot but engage little are at high churn risk (poor value perception).
# Conversely, highly engaged low-paying customers are sticky. This ratio captures that dynamic.

df['engagement_per_dollar'] = (
    (df['internet_usage_gb'] * 2 + df['call_minutes'] / 30) / (df['monthly_bill'] + 1)
)

# Additional creative feature: 'contract_flexibility_risk'
# Monthly contracts with high support tickets = maximum churn risk
df['contract_flexibility_risk'] = (
    (df['contract_type'] == 'Monthly').astype(int) * 
    (df['support_tickets'] > 0).astype(int) * 
    (df['tenure_months'] < 6).astype(int)
)

print("✅ Exercise 4 Complete: Creative features created!")
print(f"\n💡 Feature 1: engagement_per_dollar")
print(f"   Measures usage engagement relative to monthly spend.")
print(f"   Average: {df['engagement_per_dollar'].mean():.3f}")
print(f"   Churn correlation: {df['engagement_per_dollar'].corr(df['churn']):.4f}")

print(f"\n💡 Feature 2: contract_flexibility_risk")
print(f"   Flags new monthly customers with support issues (highest risk profile).")
print(f"   Customers flagged: {df['contract_flexibility_risk'].sum()}")
print(f"   Churn rate among flagged: {df[df['contract_flexibility_risk']==1]['churn'].mean():.1%}")
print(f"   Churn rate among unflagged: {df[df['contract_flexibility_risk']==0]['churn'].mean():.1%}")

# Display
df[['monthly_bill', 'internet_usage_gb', 'call_minutes', 'engagement_per_dollar', 
    'contract_type', 'support_tickets', 'tenure_months', 'contract_flexibility_risk']].head(10)